# Demo: Bayesian Turbine Projections — US Pipeline

This notebook demonstrates the complete pipeline for the United States,
from data loading through Bayesian fitting to energy yield estimation.

**Prerequisites:** Run `uv sync --all-extras` and place USWTDB data in `data/raw/US/`.

For a quick demo without raw data, this notebook uses the pre-processed
annual aggregates included in the repository.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from turbine_projections.config import REGION_CONFIGS
from turbine_projections.bayesian_model import fit_bayesian_logistic
from turbine_projections.energy_yield import compute_aep_distribution

print('Imports OK')

## 1. Load and Explore Data

In [ ]:
# Load pre-processed US annual aggregates
us_annual = pd.read_csv('../data/processed/us_annual.csv')
print(f'US annual data: {len(us_annual)} years, {us_annual.year.min()}–{us_annual.year.max()}')
us_annual.tail(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(axes, 
    ['hub_height_m_median', 'rotor_diameter_m_median', 'specific_power_wm2_median'],
    ['Hub Height [m]', 'Rotor Diameter [m]', 'Specific Power [W/m²]']):
    ax.plot(us_annual['year'], us_annual[col], 'o-', markersize=4)
    ax.set_xlabel('Year')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

fig.suptitle('US Wind Turbine Trends (Annual Medians)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Bayesian Logistic Fit

We fit the hub height model as a demonstration. The full pipeline
fits all three metrics (hub height, rotor diameter, specific power)
for all three regions.

**Note:** This uses a reduced configuration (2 chains, 1000 draws)
for speed. The production fits use 10 chains × 4000 draws.

In [ ]:
# Load full US clean data for fitting (if available)
import os

us_data_path = '../data/processed/us_clean.csv'
if os.path.exists(us_data_path):
    us_clean = pd.read_csv(us_data_path)
    # Stratified subsample to N=2000 (production setting)
    us_sample = us_clean.groupby('year', group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, int(2000 * len(x) / len(us_clean)))),
                           random_state=42)
    ).reset_index(drop=True)
    print(f'Subsampled: {len(us_sample)} from {len(us_clean)}')
else:
    print('us_clean.csv not found. Run scripts/02_prepare_data.py first,')
    print('or download USWTDB from https://eerscmap.usgs.gov/uswtdb/')
    us_sample = None

In [ ]:
if us_sample is not None:
    # Get US hub height config
    us_cfg = REGION_CONFIGS['US']
    
    # Fit with reduced settings for demo
    trace, model = fit_bayesian_logistic(
        years=us_sample['year'].values,
        values=us_sample['hub_height_m'].values,
        metric='hub_height',
        region_config=us_cfg,
        n_chains=2,
        n_draws=1000,
        target_accept=0.95,
        random_seed=42,
    )
    print('Fit complete!')
else:
    print('Skipping fit (no data). See above.')

## 3. Projections and Uncertainty

Extract posterior projections for 2030 and 2055.

In [ ]:
if us_sample is not None and trace is not None:
    import arviz as az
    
    # Summary statistics
    summary = az.summary(trace, hdi_prob=0.95)
    print(summary)
    
    # Extract posterior samples for projection
    posterior = trace.posterior
    L = posterior['L'].values.flatten()
    k = posterior['k'].values.flatten()
    t0 = posterior['t0'].values.flatten()
    y0 = posterior['y0'].values.flatten()
    
    for target_year in [2030, 2055]:
        pred = y0 + L / (1 + np.exp(-k * (target_year - t0)))
        print(f'\nUS Hub Height {target_year}:')
        print(f'  Median: {np.median(pred):.1f} m')
        print(f'  95% CI: [{np.percentile(pred, 2.5):.1f}, {np.percentile(pred, 97.5):.1f}] m')

## 4. Next Steps

The full pipeline (scripts 03–07) extends this to:

- All three metrics × three regions (9 fits)
- Hindcast benchmarking against linear, quadratic, and MLE models
- Prior sensitivity analysis (3 prior sets × 9 fits = 27 fits)
- Climate-aware energy yield with GOWIRES Weibull parameters
- Variance decomposition (technology vs. climate, scenario vs. GCM ensemble)

See the paper and `scripts/` for details.